In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch
from google.colab import files
import os
filename = "Q2_20230202_majority 1.csv"

if not os.path.exists(filename):
  print(f"'{filename}' not found. Please upload the file.")
  uploaded = files.upload()
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("Q2_20230202_majority 1.csv")

# Load sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute embeddings
tweet_texts = df['tweet'].tolist()
embeddings = model.encode(tweet_texts, convert_to_tensor=True, show_progress_bar=True)

# Save embeddings for reuse
torch.save(embeddings, 'tweet_embeddings.pt')


'Q2_20230202_majority 1.csv' not found. Please upload the file.


Saving Q2_20230202_majority 1.csv to Q2_20230202_majority 1.csv


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/180 [00:00<?, ?it/s]

In [2]:
def retrieve_similar_examples(new_tweet, k=5, balanced=True):
    new_embedding = model.encode([new_tweet], convert_to_tensor=True)
    all_embeddings = torch.load('tweet_embeddings.pt')

    # Cosine similarity
    cosine_scores = cosine_similarity(new_embedding.cpu(), all_embeddings.cpu())[0]

    df['similarity'] = cosine_scores
    if balanced:
        # Sample roughly k/3 per class
        examples = []
        for cls in ['in-favor', 'against', 'neutral-or-unclear']:
            subset = df[df['label_majority'] == cls].nlargest(k // 3, 'similarity')
            examples.extend(subset[['tweet', 'label_majority']].to_dict('records'))
    else:
        examples = df.nlargest(k, 'similarity')[['tweet', 'label_majority']].to_dict('records')

    return examples


In [3]:
def create_flan_prompt(new_tweet, retrieved_examples):
    header = (
        'Determine the stance of the following tweet about COVID-19 vaccines. '
        'Please answer with exactly one of: "in-favor", "against", "neutral-or-unclear".\n\n'
        'Here are examples:\n'
    )
    example_lines = [
        f'Tweet: "{ex["tweet"]}" → {ex["label_majority"]}' for ex in retrieved_examples
    ]
    ending = f'\nNow classify this tweet: "{new_tweet}"'

    return header + "\n".join(example_lines) + ending


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 (base or large depending on resources)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to('cuda')

def predict_with_flan(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to('cuda')
    outputs = model_t5.generate(**inputs, max_new_tokens=10)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
df = df.sample(100)

In [ ]:
df['label_pred'] = None

for i, row in df.iterrows():
    tweet = row['tweet']
    retrieved = retrieve_similar_examples(tweet, k=6)
    prompt = create_flan_prompt(tweet, retrieved)
    prediction = predict_with_flan(prompt)
    df.at[i, 'label_pred'] = prediction
